In [ ]:
%cd ../..
import os
import torch
import polars as pl
from omegaconf import OmegaConf
import matplotlib.pyplot as plt
from einops import rearrange

from dinov2.inference import build_model
from evaluation import *

In [ ]:
data_path = "/mnt/typhon/data/AI/DeepRDT/DeepRDT_rectum/Deep_rectum_crop"

metadata = pl.read_csv(os.path.join(data_path, "dfRectum_cropDetails_wClinic.csv"))
metadata.head()

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/f/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

In [ ]:
def load_img(patient_id):
    hu_min, hu_max = -900, 700

    img_path = os.path.join(data_path, f"torch_pth/{patient_id}.pth")
    img = torch.load(img_path)

    hu = (img/255.0) * (hu_max - hu_min) + hu_min
    hu = torch.where(hu > -800, hu, -1000)

    return hu

In [ ]:
idx = 1
row = metadata.row(idx)
patient_id = row[0]
label = row[-1]
img = load_img(patient_id)

plt.imshow(img[50], cmap="gray")
plt.colorbar()
plt.show()
print(img.shape)

In [ ]:
img_size = 504
channels = 10
patch_size = 14
patch_dim = img_size // patch_size

test_img = img[49:51,40:140,40:140]
test_img = test_img.unsqueeze(0).unsqueeze(0)
test_img = torch.nn.functional.interpolate(
    test_img, size=(channels, img_size, img_size), mode="trilinear"
).squeeze(0)

with torch.inference_mode():
    with autocast_ctx():
        features = model.forward_features(test_img.cuda())
features = {k: v.cpu() for k, v in features.items() if isinstance(v, torch.Tensor)}
patch_features = features["x_norm_patchtokens"]
patch_features = rearrange(patch_features, "1 (x y) d -> x y d", x=patch_dim, y=patch_dim)
patch_features.shape

In [ ]:
plot_patch_similarity(patch_features, test_img, ref_x = 11, ref_y = 23, thresh=0.0)

In [ ]:
pca_img = pca_feature_map(patch_features, (0,1,2))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

im1 = ax1.imshow(test_img[0,5], cmap='gray', vmin=-1000, vmax=1000)
fig.colorbar(im1, ax=ax1)
ax1.set_title('CT Image')

im2 = ax2.imshow(pca_img)
ax2.set_title('Colorized Feature Map')

plt.tight_layout()
plt.show()